#### 1. 라이브러리 임포트

In [5]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import re
import contractions
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import optimizers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


I0000 00:00:1780513911.620314   34858 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780513911.649235   34858 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780513912.491659   34858 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


#### 2. 시드 고정

In [6]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

#### 3. 데이터 로드

In [22]:
# parquet 형식 데이터 로드
df = pd.read_parquet('final_data.parquet')

# 데이터 크기 확인
print(f"데이터 크기: {df.shape}")

# 데이터 비율 확인
print(f"데이터 비율: {df['fake'].value_counts()}")

# 샘플 데이터 확인
df.sample(5)

데이터 크기: (70000, 7)
데이터 비율: fake
1    35000
0    35000
Name: count, dtype: int64


,review_text,fake,basic_linguistic_list,readability_list,sentiment_list,behavioral_list,clean_text
46626,"Glasserie is incredible, I don't even know whe...",1,"[266.0, 179.0, 13.0, 1015.0, 803.0, 25.0, 125....","[11.05125369901198, 67.14079501504084, 7.31519...","[0.3542929292929293, 0.546506734006734, 2.0, 0...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",glasserie is incredible i do not even know whe...
23592,This is just for the drinks part of the menu a...,0,"[67.0, 58.0, 5.0, 280.0, 212.0, 2.0, 51.0, 12....","[6.742157984588678, 95.8747931034483, 2.768482...","[0.2795454545454545, 0.4818181818181819, 0.0, ...","[64.0, 1801.0, 28.58730158730159, 38.432863508...",this is just for the drinks part of the menu a...
39954,very good place especially the zatziki snd sou...,1,"[14.0, 8.0, 1.0, 51.0, 44.0, 2.0, 4.0, 2.0, 1....","[0.0, 50.66500000000002, 8.180000000000003, 13...","[0.455, 0.8900000000000001, 0.0, 0.0, 1.0, 0.0...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0]",very good place especially the zatziki snd sou...
36168,This little corner place offers a delicious va...,1,"[57.0, 42.0, 3.0, 223.0, 177.0, 3.0, 31.0, 9.0...","[8.841846274778883, 77.81071428571428, 5.88428...","[0.196875, 0.53125, 0.0, 1.0, 5.0, 2.0, 0.0]","[4.0, 8.0, 2.6666666666666665, 2.8867513459481...",this little corner place offers a delicious va...
9543,5 years living in Astoria and just until today...,1,"[45.0, 29.0, 2.0, 169.0, 134.0, 3.0, 17.0, 8.0...","[10.12575670159684, 61.89000000000001, 8.35333...","[1.0, 0.95, 0.0, 0.0, 2.0, 0.0, 0.0]","[4.0, 1098.0, 366.0, 359.01671270290467, 0.0, ...",5 years living in astoria and just until today...


#### 4. 텍스트 전처리

#### - 텍스트 정제 함수

In [23]:
def clean_text_keep_words(text, min_words=3):

    text = str(text)

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # URL 제거
    text = re.sub(r"<[^>]+>", " ", text)                # HTML/XML 태그 제거

    try:
        text = contractions.fix(text)                   # 축약어 복원
    except Exception:
        return None                                     # contractions 처리 오류 행 제거

    # text = re.sub(r"[^A-Za-z0-9\s]", " ", text)         # 특수문자 제거
    text = re.sub(r"\.(?!\s|$)", " ", text)       # 문장 끝이 아닌 점 제거
    text = re.sub(r"[^A-Za-z0-9\s.]", " ", text)  # 나머지 특수문자 제거
    text = text.lower()                                 # 소문자 변환
    text = re.sub(r"\s+", " ", text).strip()            # 공백 정리

    if len(text.split()) < min_words:                   # 최소 단어 수 필터링
        return None

    return text

#### - 텍스트 변환 함수

In [24]:
def preprocess_and_stem_lemmatize(text):

    # 1. 텍스트 토큰화
    tokens = word_tokenize(text)

    # 2. 불용어 제거 (대소문자 무시)
    filtered_tokens = [word for word in tokens if word.lower() not in stopwords.words('english')]

    # 3. 어간 추출
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]

    # 4. 원형 복원
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]

    # 8. 공백으로 연결된 문자열로 반환
    return ' '.join(stemmed_tokens)

In [25]:
# 데이터프레임 복사
df_clean_text = df.copy()

# 텍스트 정제 함수 적용
df_clean_text["clean_text"] = df_clean_text["review_text"].astype(str).progress_apply(clean_text_keep_words)

# None 값 제거
df_clean_text = df_clean_text.dropna(subset=["clean_text"]).reset_index(drop=True)

# 텍스트 변환 함수 적용
df_clean_text['clean_text'] = (df_clean_text['clean_text'].progress_apply(preprocess_and_stem_lemmatize))


# 결과 확인
print(f"텍스트 정제 전 데이터 수: {len(df):,}")
print(f"텍스트 정제 후 데이터 수: {len(df_clean_text):,}")
print(f"제거된 데이터 수: {len(df) - len(df_clean_text):,}")

# 결과 샘플 확인
df_clean_text[["review_text", "clean_text"]].sample(5)

100%|██████████| 70000/70000 [15:08<00:00, 77.02it/s] 

텍스트 정제 전 데이터 수: 70,000
텍스트 정제 후 데이터 수: 70,000
제거된 데이터 수: 0


,review_text,clean_text
64884,WHOA. The owners of the place recognized my fa...,whoa . owner place recogn face yelp profil rev...
63754,I usually eat at this place and I can say thei...,usual eat place say pizza realli good . spamon...
51346,Ambience and location is very good. Can't comp...,ambienc locat good . complain food strike . mi...
45516,Public is a classy SoHo brunch spot that offer...,public classi soho brunch spot offer sex citi ...
26172,I finally decided to venture over to Lombardi'...,final decid ventur lombardi last week disappoi...


#### 5. Train / Validation / Test 분할

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    df_clean_text["clean_text"], df_clean_text["fake"].values.astype("float32"),
    test_size=0.2, stratify=df_clean_text["fake"], random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.125, stratify=y_train, random_state=SEED
)

print(f"Train: {len(X_train):,} / Val: {len(X_val):,} / Test: {len(X_test):,}")


Train: 49,000 / Val: 7,000 / Test: 14,000


#### 6. Tokenize

In [27]:
MAX_WORDS = 10000
MAX_LEN = 128

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_seq = tokenizer.texts_to_sequences(X_train)
val_seq   = tokenizer.texts_to_sequences(X_val)
test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(val_seq,   maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

# 데이터 크기 출력
print("X_train_pad:", X_train_pad.shape)
print("X_val_pad:", X_val_pad.shape)
print("X_test_pad:", X_test_pad.shape)

print("vocab_size:", vocab_size)

X_train_pad: (49000, 128)
X_val_pad: (7000, 128)
X_test_pad: (14000, 128)
vocab_size: 10000


In [28]:
print(len(tokenizer.word_index) + 1)
print(min(MAX_WORDS, len(tokenizer.word_index) + 1))
print(f"OOV 비율: {sum(token == tokenizer.word_index.get('<OOV>') for seq in train_seq for token in seq) / sum(len(seq) for seq in train_seq):.4f}")

35340
10000
OOV 비율: 0.0177


#### 7. CNN 함수

In [29]:
def build_cnn():
    text_input = Input(shape=(MAX_LEN,))
    x = Embedding(input_dim=vocab_size, output_dim=128)(text_input)
    x = Conv1D(filters=64, kernel_size=3, activation="relu")(x)
    x = GlobalMaxPooling1D()(x)

    x = Dense(2048, activation="relu")(x)
    x = Dense(1024, activation="relu")(x)
    x = Dense(512, activation="relu")(x)
    x = Dense(256, activation="relu")(x)
    x = Dense(128, activation="relu")(x)

    output = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=text_input, outputs=output)

    return model

#### 8. 모델 생성

In [30]:
model = build_cnn()

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 128, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 126, 64)        │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 2048)           │       133,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,225,089 (16.12 MB)

 Trainable params: 4,225,089 (16.12 MB)

 Non-trainable params: 0 (0.00 B)

#### 9. 모델 컴파일

In [31]:
model.compile(optimizer=optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])

#### 10. 콜백 설정

In [32]:
early_stopping = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

#### 11. 모델 학습

In [35]:
history = model.fit(
    x=X_train_pad, y=y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20, batch_size=32,
    callbacks=early_stopping,
)


Epoch 1/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 1s 955us/step - accuracy: 0.7321 - loss: 0.5441 - val_accuracy: 0.6409 - val_loss: 0.6781
Epoch 2/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 1s 948us/step - accuracy: 0.7939 - loss: 0.4527 - val_accuracy: 0.6240 - val_loss: 0.8548
Epoch 3/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 1s 946us/step - accuracy: 0.8738 - loss: 0.3078 - val_accuracy: 0.6073 - val_loss: 1.1067
Epoch 4/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 1s 937us/step - accuracy: 0.9155 - loss: 0.2139 - val_accuracy: 0.6001 - val_loss: 1.3016
Epoch 5/20
1532/1532 ━━━━━━━━━━━━━━━━━━━━ 1s 947us/step - accuracy: 0.9311 - loss: 0.1702 - val_accuracy: 0.6140 - val_loss: 1.1449


#### 12. 예측 및 성능 계산

In [36]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 369us/step
Accuracy : 0.6369
Precision: 0.6091
Recall   : 0.7639
F1 Score : 0.6778
